In [1]:
from app.services.pipeline import coletar_dados

from time import perf_counter

username = "felipe.cruz"
password = "#Gladoscruz.9851"
analise = "compra por necessidade"

dfs = coletar_dados(username, password, analise)

['Item', 'Familia', '']
Extractor criado com sucesso!
consCAPAS LE concluída em 21.45s (1 / 102)
apoio_compras_FNC concluída em 24.96s (2 / 102)
apoio_compras_VRG concluída em 32.75s (3 / 102)
apoio_compras_VSN concluída em 32.97s (4 / 102)
ordens_ESTOF UL concluída em 41.75s (5 / 102)
consMETALURGIA concluída em 19.56s (6 / 102)
apoio_compras_CST concluída em 44.30s (7 / 102)
ordens_KIT concluída em 45.89s (8 / 102)
consUL ACABA01 concluída em 48.13s (9 / 102)
ordens_CAPAS LE concluída em 47.39s (10 / 102)
ordens_LE FABRI02 concluída em 30.26s (11 / 102)
consPLANEJADOS concluída em 53.12s (12 / 102)
apoio_compras_CSD concluída em 20.71s (13 / 102)
apoio_compras_NEC concluída em 26.22s (14 / 102)
ordens_CC FABRI03 concluída em 11.12s (15 / 102)
ordens_ACESSORIOS concluída em 12.87s (16 / 102)
ordens_CC ACABA01 concluída em 16.49s (17 / 102)
consFIBRA concluída em 8.86s (18 / 102)
consCC FABRI01 concluída em 19.78s (19 / 102)
ordens_PLANEJADOS concluída em 17.57s (20 / 102)
consATELIE c

In [2]:
import pickle

with open("snapshot_dfs.pkl", "wb") as f:
    pickle.dump(dfs, f)

print("Snapshot salvo.")



Snapshot salvo.


In [1]:
import pickle

dfs = {}
with open("snapshot_dfs.pkl", "rb") as f:
    dfs = pickle.load(f)

In [ ]:
import pandas as pd
import networkx as nx

print(f"{dfs.keys()}\n")


def sanitizar_dataframe(df, limite=0.8):
    df = df.copy()

    for col in df.columns:
        serie = df[col].astype(str).str.strip()

        tentativa_data = pd.to_datetime(
            serie, errors="coerce", dayfirst=True, format="%d/%m/%Y"
        )
        if tentativa_data.notna().mean() > limite:
            df[col] = tentativa_data
            continue

        serie_num = serie.str.replace(".", "", regex=False).str.replace(
            ",", ".", regex=False
        )
        tentativa_num = pd.to_numeric(serie_num, errors="coerce")
        if tentativa_num.notna().mean() > limite:
            df[col] = tentativa_num
            continue

        df[col] = serie.replace({"": None})

    return df


def calc_data(dfs):

    ## Ajuste Ordens ##
    ordens = sanitizar_dataframe(dfs.get("ordens"))
    ordens = ordens[
        [
            "Cliente",
            "Fábrica",
            "Ordem",
            "Pedido",
            "Item",
            "Saldo",
            "Representante",
            "Entrega Pedido",
            "Data Abertura",
        ]
    ]
    ordens = ordens.rename(columns={"Ordem": "Ordem Prod", "Saldo": "Saldo Prod"})

    colunas = ["Entrega Pedido", "Data Abertura"]
    for col in colunas:
        # Remove o ponto e garante que a coluna seja tratada como string
        ordens[col] = (
            ordens[col]
            .astype(str)
            .str.strip()
            .str.replace(r"[^\d]", "", regex=True)  # remove tudo que não for número
            .pipe(lambda s: pd.to_datetime(s, format="%d%m%Y", errors="coerce"))
        )
    ## Ajuste Consumo ##
    consumo = sanitizar_dataframe(dfs.get("cons"))

    consumo["Item"] = consumo["Item"].str.split("-").str[0].str.strip()
    print(consumo.columns)
    consumo = consumo[["Item", "Baixa", "Consumo", "Local Prod.", "OP", "Familia"]]
    consumo = consumo.rename(columns={"OP": "Ordem Cons"})

    ######## Item Pai ##########

    consumo["item_pai"] = consumo["Ordem Cons"].map(
        ordens.set_index("Ordem Prod")["Item"]
    )
    ######## Importa dados ##########
    consumo = consumo.merge(ordens[["Item", "Ordem Prod"]], on="Item", how="left")

    consumo = consumo.merge(
        ordens[
            [
                "Cliente",
                "Fábrica",
                "Ordem Prod",
                "Pedido",
                "Saldo Prod",
                "Representante",
                "Entrega Pedido",
                "Data Abertura",
            ]
        ].rename(columns={"Ordem Prod": "Ordem", "Saldo Prod": "Saldo"}),
        left_on="Ordem Cons",
        right_on="Ordem",
        how="left",
    )

    ######## Ordena as colunas ##########

    consumo = consumo[
        [
            "Ordem Prod",
            "Item",
            "Consumo",
            "Ordem Cons",
            "item_pai",
            "Saldo",
            "Pedido",
            "Representante",
            "Entrega Pedido",
            "Local Prod.",
            "Familia",
            "Cliente",
            "Fábrica",
            "Ordem",
            "Data Abertura",
            "Baixa",
        ]
    ]
    ######## Calculos Baseados em estoque ##########
    estoque = sanitizar_dataframe(dfs.get("estoque"))
    consumo["estoque"] = (
        consumo["Item"].map(estoque.groupby("Item")["Qtde."].sum()).fillna(0)
    )

    csv_path = "CSV/"
    consumo.to_excel(csv_path + "consumo.xlsx", index=False)


calc_data(dfs)

dict_keys(['cons', 'apoio_compras', 'ordens', 'estoque'])

Index(['Tipo', 'Grupo', 'Item', 'Local Estoque', 'Baixa', 'Situação',
       'Den. Item', 'Consumo', 'Local Prod.', 'OP', 'Familia', 'Família'],
      dtype='str')
             Item                                          Descrição  FAM  \
0  10013050001442  SOFA PF 013 ESTR MAD - TECIDO LINHO LISO 1275 ...  013   
1  10026050003056  PUFF 026 DIAM 81 X 36H CM ESTR MAD - TECIDO CO...  026   
2  10026050003052  PUFF 026 DIAM 81 X 36H CM ESTR MAD - TECIDO LI...  026   
3  10026470002073  SOFA PF 026 47 ESTR MAD - TECIDO COURIS LISO 2...  026   
4  10026470003467  PUFF 026 DIAM 47 X 36H CM ESTR MAD - TECIDO LI...  026   

  Sit.     Local - Sit  Qtde. Unidade  Qt.Reser   PE Endereço Local Prod.  \
0    I  TERCEIROS  - L    1.0      UN       0.0  NAO        .  LE FABRI02   
1    A  175252     - L    1.0      UN       0.0  NAO        .  LE FABRI02   
2    A  174353     - L    1.0      UN       1.0  NAO        .  LE FABRI02   
3    I